# Analysis -> synthesis reconstruction

This notebook tests the full round trip: `PolyphaseAnalysisChannelizer.process` followed by
`PolyphaseSynthesisChannelizer.process` with matching parameters. For a channelizer/synthesizer pair this
should reconstruct the input up to a known delay and scale factor ("near-perfect reconstruction").

**Status:** the existing unit test (`tests/test_channelizer.py`) only checks that the round-trip output has
the right shape and is finite — it never checks that the output actually resembles the input. This notebook
adds that check and currently demonstrates that reconstruction is **broken**: the output is essentially
uncorrelated with the input, not just imperfect. This is a separate, still-open issue from the
channel-to-frequency mapping bug fixed in `01_prototype_filter_and_channel_mapping.ipynb`.

In [ ]:
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import numpy as np
import matplotlib.pyplot as plt

from pypoly import PolyphaseAnalysisChannelizer, PolyphaseSynthesisChannelizer

%matplotlib inline

In [ ]:
def best_delay_scale_error(x: np.ndarray, x_hat: np.ndarray, max_delay: int) -> tuple[int, complex, float]:
    """Search delays for the best linear (delay + complex scale) match between x and x_hat.

    Returns (best_delay, best_scale, normalized_error). normalized_error near 0 means good
    reconstruction; near 1 means x_hat is essentially uncorrelated with x.
    """
    best = None
    for d in range(max_delay):
        a = x[: len(x) - d]
        b = x_hat[d : len(x)]
        n = min(len(a), len(b))
        a, b = a[:n], b[:n]
        denom = np.vdot(a, a)
        if denom == 0:
            continue
        scale = np.vdot(a, b) / denom
        err = np.linalg.norm(b - scale * a) / np.linalg.norm(b)
        if best is None or err < best[2]:
            best = (d, scale, err)
    return best

In [ ]:
M = 8
TAPS_PER_CHANNEL = 16

analysis = PolyphaseAnalysisChannelizer.from_design(num_channels=M, taps_per_channel=TAPS_PER_CHANNEL)
synthesis = PolyphaseSynthesisChannelizer.from_design(num_channels=M, taps_per_channel=TAPS_PER_CHANNEL)

rng = np.random.default_rng(0)
x = rng.normal(size=4096) + 1j * rng.normal(size=4096)

y = analysis.process(x)
x_hat = synthesis.process(y)

delay, scale, err = best_delay_scale_error(x, x_hat, max_delay=300)
print(f"best delay: {delay} samples, scale: {scale:.4g}, normalized error: {err:.4f}")
print("(normalized error near 0 = good reconstruction; near 1 = uncorrelated)")

## Visualizing the mismatch

Plot the (delay- and scale-aligned) reconstruction against the original input over a short window. If
reconstruction were working we'd expect the two traces to overlap closely.

In [ ]:
window = slice(0, 200)
a = x[: len(x) - delay][window]
b = (scale * x_hat[delay : len(x)])[window]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(a.real, label="x (original, real part)")
ax.plot(b.real, label="x_hat (aligned, real part)", linestyle="--")
ax.set_title(f"Analysis -> synthesis round trip (normalized error = {err:.3f})")
ax.set_xlabel("sample")
ax.legend()
ax.grid(True, alpha=0.3)

## Impulse response of the full pipeline

Push a single impulse through the full analysis -> synthesis pipeline. A working channelizer/synthesizer
pair should produce one concentrated peak near the combined filter group delay
(`(taps_per_channel * num_channels)` samples for this causal FIR pair). Instead we see energy smeared into
multiple peaks spaced by `M` samples -- a signature of un-cancelled decimation/interpolation images rather
than a single reconstructed impulse.

In [ ]:
impulse = np.zeros(512, dtype=complex)
impulse[100] = 1.0

y_imp = analysis.process(impulse)
imp_hat = synthesis.process(y_imp)

fig, ax = plt.subplots(figsize=(9, 4))
ax.stem(np.abs(imp_hat))
ax.set_title("|output| for an impulse input at n=100 (full analysis->synthesis pipeline)")
ax.set_xlabel("sample")
ax.set_ylabel("magnitude")
ax.grid(True, alpha=0.3)

## Takeaway

The analysis and synthesis kernels are not actually inverses of one another as currently indexed -- the
synthesis commutator's output ordering does not mirror the analysis commutator's input ordering
(`x[nM - p]`), so the M-fold images created by decimation in the analysis stage are not properly cancelled
by the interpolation in the synthesis stage. This needs a follow-up fix to `_synthesis_polyphase_kernel`
(or the phase/commutator convention it assumes) before `PolyphaseSynthesisChannelizer` can be considered
correct.